# PREDICT-Play | 01. Preprocessing and simulation
 See [Preprocessing.MD](../../docs/part_b/Preprocessing.MD).

The business event is **150 distinct households each watching at least half the runtime in a 12-month licence period**. MovieLens does not measure this event. We construct it using an explicitly assumed rating-to-duration simulation.

All source years are pooled, with interactions reassigned to the synthetic period **1 September 2025 to 31 August 2026**. This is NOT a temporal backtest. No restriction is placed on the age of a film. Renewing the exercise annually does not, by itself, validate a next-year forecast.

## Imports

In [1]:
# Import the preprocessing routines and create the raw, processed and sealed areas.
from getpass import getpass
import pandas as pd
import numpy as np
import pandas as pd
import requests
import zipfile
from pathlib import Path
from datetime import datetime, timedelta, timezone

## Import Data

In [2]:
url = "https://files.grouplens.org/datasets/movielens/ml-20m.zip"

response = requests.get(url)

with open("ml-20m.zip", "wb") as file:
    file.write(response.content)

with zipfile.ZipFile("ml-20m.zip", "r") as zip_ref:
    zip_ref.extractall("data")

In [3]:
movies = pd.read_csv("data/ml-20m/movies.csv")
links = pd.read_csv("data/ml-20m/links.csv")
ratings = pd.read_csv("data/ml-20m/ratings.csv")

## 2. Create catalogue and Platform data
The official archive is cached. Only `movies.csv`, `links.csv` and `ratings.csv` are used.
The split is approximately 40% **distinct canonical titles** on-platform and 60% acquisition candidates; it is not a split of individual rating rows. The catalogue is selected without using its outcomes.

In [4]:
# Build the title catalogue by joining MovieLens titles to their external IDs.

catalogue = movies.merge(links,on="movieId",how="left")

# Extract the release year from the MovieLens title.
catalogue["release_year"] = (catalogue["title"].str.extract(r"\((\d{4})\)$")[0].astype("Int64"))

# Create a clean title without the release year.
catalogue["canonical_title"] = (catalogue["title"].str.replace(r"\s*\(\d{4}\)$", "", regex=True).str.strip())

# Keep the TMDB ID as a nullable integer.
catalogue["tmdb_id"] = catalogue["tmdbId"].astype("Int64")

catalogue["title_id"] = ("STAD-MOV-"+ catalogue["movieId"].astype(str).str.zfill(6))

# Remove the original external-ID columns now that they have been standardised.
catalogue = catalogue.drop(columns=["imdbId", "tmdbId"])

catalogue.head()

,movieId,title,genres,release_year,canonical_title,tmdb_id,title_id
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,Toy Story,862,STAD-MOV-000001
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,Jumanji,8844,STAD-MOV-000002
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,Grumpier Old Men,15602,STAD-MOV-000003
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,Waiting to Exhale,31357,STAD-MOV-000004
4,5,Father of the Bride Part II (1995),Comedy,1995,Father of the Bride Part II,11862,STAD-MOV-000005


In [5]:
# Randomly split distinct titles into the simulated StadioChoice platform and the acquisition catalogue.

rng = np.random.default_rng(42)

shuffled_movie_ids = rng.permutation(catalogue["movieId"])

n_platform = int(len(shuffled_movie_ids) * 0.40)

platform_movie_ids = shuffled_movie_ids[:n_platform]

catalogue["pool"] = np.where(
    catalogue["movieId"].isin(platform_movie_ids),
    "platform",
    "acquisition_catalogue"
)

## 3. Get  TMDB details and credits for BOTH title pools

TMDB daily exports contain valid TMDB movie IDs and a small number of high-level attributes. They are used here instead of making thousands of individual API requests. API Requests can be used in future if we want to make use of the extensive ratings data present on TMDB

In [6]:
from getpass import getpass

tmdb_api_key = getpass("Enter your TMDB API key: ")

Enter your TMDB API key: ··········


In [7]:

# Use yesterday's date because the daily TMDB export is produced each morning and yesterday's file should already be complete.
export_date = datetime.now(timezone.utc).date() - timedelta(days=1)
date_string = export_date.strftime("%m_%d_%Y")
tmdb_export_url = (f"https://files.tmdb.org/p/exports/"f"movie_ids_{date_string}.json.gz")


response = requests.get(tmdb_export_url)
response.raise_for_status()

export_path = "data/tmdb_movie_ids.json.gz"

with open(export_path, "wb") as file:
    file.write(response.content)


# Convert JSON to CSV
tmdb_export = pd.read_json( export_path, lines=True, compression="gzip")

In [8]:
# Join the MovieLens catalogue to the TMDB daily export.

tmdb_export = tmdb_export.rename(columns={"id": "tmdb_id","popularity": "tmdb_popularity"})

# Make sure the TMDB IDs use the same datatype.
tmdb_export["tmdb_id"] = tmdb_export["tmdb_id"].astype("Int64")
catalogue["tmdb_id"] = catalogue["tmdb_id"].astype("Int64")


# Create the master enriched content catalogue.
content_catalogue = catalogue.merge(tmdb_export,on="tmdb_id",how="left")


# Flag whether the MovieLens TMDB ID was found in the current TMDB export.
content_catalogue["tmdb_available"] = (content_catalogue["tmdb_popularity"].notna())


# Create the two title pools from the same master catalogue.
platform_catalogue = content_catalogue[content_catalogue["pool"] == "platform"].copy()
acquisition_catalogue = content_catalogue[content_catalogue["pool"] == "acquisition_catalogue"].copy()

# Simulate StadioChoice platform viewing behaviour using TMDB popularity.

Simulation assumptions:
* Only titles in the existing "platform" pool generate viewing data.
* MovieLens userId represents an anonymised household.
* A MovieLens user-title interaction represents one synthetic viewing session.
* MovieLens rating values are NOT used.
* TMDB popularity is converted to a percentile because the raw field is strongly right-skewed.
* The popularity percentile is used as the probability that a household watches at least 50% of the title.
* A title receives target = 1 when at least 150 distinct households qualify.

TMDB popularity is used only to generate the synthetic outcome and must not later be used as a model feature.

In [9]:
# Set simulation start and end dates

simulation_start = pd.Timestamp("2025-09-01")
simulation_end = pd.Timestamp("2026-09-01")

rng = np.random.default_rng(42)

## 5. Generate viewing logs and the platform target


In [10]:

#Build the lookup for titles currently on the simulated platform.

platform_titles = platform_catalogue[["movieId","title_id","tmdb_popularity"]].copy()



# Convert TMDB popularity to a percentile.
# Raw TMDB popularity is highly right-skewed, so ranking the values gives each title a relative popularity score between approximately 0 and 1.


platform_titles["popularity_percentile"] = (platform_titles["tmdb_popularity"].rank(pct=True))


# Titles without TMDB popularity receive the median percentile.
# This keeps unresolved TMDB titles in the simulation.

platform_titles["popularity_percentile"] = (platform_titles["popularity_percentile"].fillna(0.50))


# Keep MovieLens user-title interactions belonging only to platform titles.

platform_ratings = ratings[ratings["movieId"].isin(platform_titles["movieId"])]
platform_pairs = platform_ratings[["userId", "movieId"]]
platform_pairs = platform_pairs.drop_duplicates(subset=["userId", "movieId"])
platform_pairs = platform_pairs.merge(platform_titles,on="movieId",how="left")
platform_interactions = platform_pairs.reset_index(drop=True)




## Use TMDB popularity as the viewing benchmark.

The popularity percentile becomes the probability that an observed household watches at least 50% of the title.

Example:
* 20th percentile popularity -> approximately 20% qualifying probability
* 50th percentile popularity -> approximately 50% qualifying probability
* 90th percentile popularity -> approximately 90% qualifying probability


In [11]:

platform_interactions["qualifying_probability"] = (platform_interactions["popularity_percentile"])


# Simulate whether each household watches at least 50% of the title.

platform_interactions["qualifying_household"] = (rng.random(len(platform_interactions)) < platform_interactions["qualifying_probability"])



# Simulate the proportion of the title watched.
# Qualifying households are assigned between 50% and 100%.Non-qualifying households are assigned between 5% and 49%.


qualifying_watch_fraction = rng.uniform(0.50, 1.00, size=len(platform_interactions))

non_qualifying_watch_fraction = rng.uniform(0.05, 0.49, size=len(platform_interactions))

platform_interactions["watch_fraction"] = np.where( platform_interactions["qualifying_household"],qualifying_watch_fraction,non_qualifying_watch_fraction)



# Create a neutral synthetic runtime. The TMDB daily export does not contain runtime.
# A fixed 120-minute runtime is therefore used only to convert the simulated watch fraction into watch-duration seconds.

platform_interactions["title_runtime_seconds"] = 120 * 60

platform_interactions["watch_duration_seconds"] = (
    platform_interactions["title_runtime_seconds"]
    *
    platform_interactions["watch_fraction"]
).round().astype(int)



# Give each synthetic viewing session a timestamp within the 12-month simulation period.


simulation_seconds = int((simulation_end - simulation_start).total_seconds())

random_seconds = rng.integers(0,simulation_seconds,size=len(platform_interactions))

platform_interactions["event_timestamp"] = (
    simulation_start
    +
    pd.to_timedelta(
        random_seconds,
        unit="s"
    )
)




## Build the synthetic data tables and aggregate for modelling

In [12]:
# Build the synthetic platform viewing-session table.


platform_sessions = pd.DataFrame({
    "household_id": platform_interactions["userId"],
    "session_id": np.arange(
        1,
        len(platform_interactions) + 1
    ),
    "event_timestamp": platform_interactions["event_timestamp"],
    "movieId": platform_interactions["movieId"],
    "title_id": platform_interactions["title_id"],
    "event_type": "Play",
    "watch_duration_seconds": platform_interactions["watch_duration_seconds"],
    "title_runtime_seconds": platform_interactions["title_runtime_seconds"]
})



# Aggregate viewing to household-title level.


household_title_totals_grouped = (platform_sessions.groupby(["household_id","movieId","title_id"],as_index=False))
household_title_totals = household_title_totals_grouped.agg(
        session_count=(
            "session_id",
            "count"
        ),
        total_watch_seconds=(
            "watch_duration_seconds",
            "sum"
        ),
        title_runtime_seconds=(
            "title_runtime_seconds",
            "first"
        )
    )


# A household qualifies when cumulative viewing reaches at least 50% of the title runtime.

household_title_totals["qualifying_household"] = (
    household_title_totals["total_watch_seconds"]
    >=
    household_title_totals["title_runtime_seconds"] * 0.50
)



# Aggregate to title level.


title_outcomes = (
    household_title_totals
    .groupby(
        [
            "movieId",
            "title_id"
        ],
        as_index=False
    )
    .agg(
        observed_households=(
            "household_id",
            "nunique"
        ),
        qualifying_households=(
            "qualifying_household",
            "sum"
        )
    )
)



# Join back to every platform title.

outcomes = (
    platform_titles[
        [
            "movieId",
            "title_id"
        ]
    ]
    .merge(
        title_outcomes,
        on=[
            "movieId",
            "title_id"
        ],
        how="left"
    )
)

outcomes[["observed_households","qualifying_households"]] = (outcomes[["observed_households","qualifying_households"]]

.fillna(0)
    .astype(int)
)




## Create the binary modelling target.
Business definition: A successful title must have at least 150 distinct households that each watched at least 50% of the title.


In [13]:

# Create the binary modelling target.


outcomes["target"] = (outcomes["qualifying_households"] >= 150).astype(int)



# Create daily viewing summaries.


platform_sessions["event_date"] = (platform_sessions["event_timestamp"].dt.date)

daily_viewing = (
    platform_sessions
    .groupby(
        [
            "event_date",
            "movieId",
            "title_id"
        ],
        as_index=False
    )
    .agg(
        viewing_events=(
            "session_id",
            "count"
        ),
        households=(
            "household_id",
            "nunique"
        ),
        watch_duration_seconds=(
            "watch_duration_seconds",
            "sum"
        )
    )
)


# Generate synthetic StadioChoice search-query data.

Simulation assumptions:
* 70% of searches are for titles currently available on the platform.
* 30% are for acquisition-catalogue titles that are not currently available.
* Searches for platform titles return results.
* Searches for acquisition titles return zero results.
* 75% of successful searches result in a click.
* Search demand varies by title using a randomly generated latent demand weight.


In [14]:
rng = np.random.default_rng(42)

n_searches = 50_000

In [15]:

# Create independent synthetic search-demand weights for titles.

search_catalogue = content_catalogue[["movieId","title_id","canonical_title","pool"]].copy()
search_catalogue["search_weight"] = rng.lognormal(mean=0,sigma=1,size=len(search_catalogue))



# Separate available and unavailable titles.

available_titles = search_catalogue[search_catalogue["pool"] == "platform"].copy()
unavailable_titles = search_catalogue[search_catalogue["pool"] == "acquisition_catalogue"].copy()


# Convert demand weights into sampling probabilities.

available_probabilities = (available_titles["search_weight"]/ available_titles["search_weight"].sum())
unavailable_probabilities = (unavailable_titles["search_weight"]/ unavailable_titles["search_weight"].sum())



# Decide whether each search is for available or unavailable content.


search_is_available = (rng.random(n_searches) < 0.70)
n_available = search_is_available.sum()
n_unavailable = n_searches - n_available


# Sample the titles being searched for.

available_samples = available_titles.sample(n=n_available,replace=True,weights=available_probabilities,random_state=42).reset_index(drop=True)

unavailable_samples = unavailable_titles.sample(n=n_unavailable,replace=True,weights=unavailable_probabilities,random_state=42).reset_index(drop=True)



# Combine the sampled searches.


available_samples["zero_results"] = False
unavailable_samples["zero_results"] = True

searches = pd.concat([available_samples,unavailable_samples],ignore_index=True)

# Shuffle so available and unavailable searches are mixed together.
searches = searches.sample(frac=1,random_state=42).reset_index(drop=True)



### Add time and events to search

In [16]:

# Create search timestamps.


search_start = simulation_end - pd.DateOffset(years=2)
search_seconds = int((simulation_end - search_start).total_seconds())
random_seconds = rng.integers(0,search_seconds,size=len(searches))
searches["search_timestamp"] = (search_start+ pd.to_timedelta(random_seconds,unit="s"))

# Create anonymised search-session IDs.
searches["session_id"] = [f"SEARCH-{i:08d}"for i in range(1, len(searches) + 1)]

#  Create the search term.
searches["search_term"] = (searches["canonical_title"].str.lower().str.strip())


# Simulate whether a returned result is clicked.

click_probability = 0.75

clicked = ((~searches["zero_results"]) &(rng.random(len(searches)) < click_probability))

searches["result_clicked"] = pd.NA
searches.loc[clicked,"result_clicked"] = searches.loc[clicked,"title_id"]


### Simulate results and create search data

In [17]:

#  Create the top-five returned results.

platform_title_ids = (available_titles["title_id"].dropna().to_numpy())

top_5_results = []

for row in searches.itertuples():

    if row.zero_results:

       top_5_results.append([])

    else:

        alternative_titles = platform_title_ids[platform_title_ids != row.title_id]
        additional_results = rng.choice(alternative_titles,size=4,replace=False).tolist()
        top_5_results.append([row.title_id] + additional_results)

searches["top_5_results_returned"] = top_5_results



# Keep the fields defined in the StadioChoice search data request.

searches = searches[[
        "session_id",
        "search_timestamp",
        "search_term",
        "result_clicked",
        "top_5_results_returned",
        "zero_results",
        "movieId"
    ]
]


## Export Data for Later Usage

In [18]:
content_catalogue.to_csv("data/content_catalogue.csv", index=False)
platform_catalogue.to_csv("data/platform_catalogue.csv", index=False)
acquisition_catalogue.to_csv("data/acquisition_catalogue.csv", index=False)
platform_sessions.to_csv("data/platform_sessions.csv", index=False)
outcomes.to_csv("data/platform_title_outcomes.csv", index=False)
searches.to_csv("data/search_queries.csv", index=False)